In [ ]:
# preparation 结果材料 → 候选出题 → 固定版本读表 → 题目与参考图文预览。
import asyncio
import base64
import html
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

# 1. 环境：DATA_ROOT 是包含 demiwtg/ 和共享 datasets/ 的工作区。
PROJECT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
sys.path.insert(0, str(PROJECT)) if str(PROJECT) not in sys.path else None
# 添加源码包的父目录，避免工作区同名外层目录被识别为 namespace package。
DEMIFLOW_PROJECT = PROJECT.parent / 'demiflow'
sys.path.insert(0, str(DEMIFLOW_PROJECT)) if str(DEMIFLOW_PROJECT) not in sys.path else None
DATA_ROOT = PROJECT.parent
os.environ['DEMIWTG_DATASETS_ROOT'] = str(DATA_ROOT)

# 长驻内核同时缓存平台入口与业务模块；先刷新 reader，再刷新消费它们的 pipeline。
from importlib import reload
from demiflow.data import api as data_api
reload(data_api)
from demiflow.data import read_api
reload(read_api)
from demiflow import data
reload(data)

from demiflow.lance.records import RecordRef
from preparation.operaters.inputs import pixels
from demiflow.operator_llm import client as model_client
from benchmark.t2i.v2.operaters import images as image_ops, authoring as authoring_ops

reload(model_client)
reload(image_ops)
reload(authoring_ops)
from benchmark.t2i.v2 import t2i_v2_benchmark_pipeline as pipeline
reload(pipeline)
config, run_pipeline = pipeline.config, pipeline.run_pipeline

# 2. 输入：文章、图片各指定一张 preparation 表及固定版本；不使用某类材料时设为 None。
# 文章只取上游清洗后的已审核正文；图片独立按已发布且 keep 的概念关系读取，不匹配配图。
# 下列版本是旧快照；新清洗规则需先由 preparation 重新导出，再填写新版本号。
ARTICLE_SOURCE = {"uri": "demiwtg/preparation/datasets/articles.lance", "version": 4}
VISUAL_SOURCE = {"uri": "demiwtg/preparation/datasets/images.lance", "version": 5}
CONCEPTS = ['莜面栲栳栳']
MODEL = 'glm/glm-5.3-flash'
RUN_ID = 'glm53_single_question_20260926_01'  # 运行名用于日志及输出分组；参数或代码修改后可直接重跑
OUTPUT_TABLE_URI = 'demiwtg/benchmark/t2i/v2/datasets/candidates__glm53_single_question_20260926_01.lance'  # 最终结果表，可直接修改。
WRITE_MODE = 'overwrite'  # 'overwrite' 覆盖目标表；'append' 追加本次结果。
RUN_DIR = DATA_ROOT / 'demiwtg/benchmark/t2i/v2/datasets' / RUN_ID
CONFIG = config(
    run=RUN_DIR, article_source=ARTICLE_SOURCE, visual_source=VISUAL_SOURCE,
    target_uri=OUTPUT_TABLE_URI, write_mode=WRITE_MODE,
    concepts=CONCEPTS, mode='modelhub', model=MODEL, max_output_tokens=16384,
    concurrency=1,  # 同时请求的概念数；例如设为 4 可并发处理四个概念
    queue_depth=1,  # 节点队列深度，限制等待处理/交付的记录缓存
    temperature=0,  # 模型采样温度

)

# 3. 运行正式 pipeline。手动执行本格会调用模型；每个概念一次，返回一道题或说明不足，不自动审题、打题或评测。
state = await asyncio.to_thread(run_pipeline, CONFIG)
display(state)

# 4. 只读本次 writer 已提交的版本；候选表每题一行，设计表保留零候选/失败原因。
TABLE_URI = state['candidates']['uri']
VERSION = state['candidates']['version']
print('待审候选表:', TABLE_URI, 'version:', VERSION)
questions = data.read_lance(TABLE_URI, version=VERSION).take_all()
DESIGNS_URI = state['designs']['uri']
DESIGNS_VERSION = state['designs']['version']
designs = data.read_lance(DESIGNS_URI, version=DESIGNS_VERSION).take_all()

# 5. 按概念展示题目与原编号材料；相同概念的参考只展示一次，点击图片可展开。
# 表格占满输出区，长文字完整显示并自动换行；改下面 width 百分比即可调整列宽。
display(HTML("""<style>
table.t2i-preview {width:100%; table-layout:fixed;}
table.t2i-preview th, table.t2i-preview td {
    text-align:left; vertical-align:top; white-space:pre-wrap;
    overflow-wrap:anywhere; max-width:none;
}
table.t2i-status th:nth-child(1) {width:20%;}
table.t2i-status th:nth-child(2) {width:80%;}
table.t2i-points th:nth-child(1) {width:45%;}
table.t2i-points th:nth-child(2) {width:55%;}
</style>"""))
for design in designs:
    display(HTML('<h3>' + html.escape(design['concept']) + '</h3>'))
    display(HTML(pd.DataFrame([{k: design[k] for k in ('status', 'reason')}])
                 .to_html(index=False, escape=True, classes='t2i-preview t2i-status')))
    # reasoning 直接从输出表读取；原生日志用于补充排查完整响应。
    reasoning = design.get('reasoning')
    if reasoning:
        display(HTML('<details><summary>模型返回的 reasoning（点击展开）</summary>'
                     + '<div style="white-space:pre-wrap">' + html.escape(reasoning) + '</div></details>'))
    else:
        display(HTML('<p>本条输出没有 reasoning；模型未返回或旧表未保存该列。</p>'))
    call = json.loads(design.get('call_json') or '{}')
    response_ref = call.get('response_ref')
    if response_ref:
        response = RecordRef.from_dict(response_ref).read(DATA_ROOT)
        body = response.get('body')
        choices = body.get('choices') or [] if isinstance(body, dict) else []
        choice = choices[0] if choices else {}
        display(HTML(pd.DataFrame([{
            'model': call.get('model'), 'reused': call.get('reused'),
            'elapsed_s': response.get('elapsed_s'), 'finish_reason': choice.get('finish_reason'),
            'usage': json.dumps(body.get('usage') or {}, ensure_ascii=False) if isinstance(body, dict) else '',
            'reasoning_chars': len(reasoning) if isinstance(reasoning, str) else None,
        }]).to_html(index=False, escape=True, classes='t2i-preview')))
        display(HTML('<details><summary>完整原始响应及调用引用（点击展开）</summary>'
                     + '<div style="white-space:pre-wrap">'
                     + html.escape(json.dumps({'response_ref': response_ref, 'response': response}, ensure_ascii=False, indent=2))
                     + '</div></details>'))
    else:
        display(HTML('<p>尚无已保存的响应；请求待处理、预算跳过或传输失败时可能没有 response_ref。</p>'))
    for question in [q for q in questions if q['concept'] == design['concept']]:
        display(HTML('<p><b>题目（待审）</b>：' + html.escape(question['instruction']) + '</p>'))
        display(HTML(pd.DataFrame(question['test_points']).rename(columns={'point': '考点', 'basis': '依据'})
                     .to_html(index=False, escape=True, classes='t2i-preview t2i-points')))
    display(HTML('<b>构题实际参考材料；开卷可提供，闭卷可省略</b>'))
    if not json.loads(design['references_json']):
        display(HTML('<p>本概念未提供参考材料，模型依据自身知识出题。</p>'))
    for ref in json.loads(design['references_json']):
        display(HTML('<p>材料 ' + str(ref['number']) + '</p>'))
        if ref['kind'] == 'text':
            display(HTML('<div style="white-space:pre-wrap">' + html.escape(ref['text']) + '</div>'))
        else:
            raw, mime, _ = pixels({'blob_ref': ref['blob_ref'], 'sha256': ref['blob_ref']['sha256']})
            url = 'data:' + mime + ';base64,' + base64.b64encode(raw).decode()
            display(HTML('<details><summary><img src="' + url + '" style="max-width:160px;max-height:160px"> 点击展开</summary>'
                         + '<img src="' + url + '" style="max-width:720px;max-height:720px"></details>'))


[T2I V2][glm53_authoring_20260926_02][+0.0s] 开始运行，等待运行锁；概念数=1，模型=glm/glm-5.3-flash，模式=modelhub，每概念一次、串行；续跑先查调用日志
[T2I V2][glm53_authoring_20260926_02][+0.1s] 已取得运行锁，检查并冻结配置及来源版本


ValueError: Run configuration, sources or code changed; use a new run name